In [1]:
import platform
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd

from vascular_superenhancement.utils.path_config import load_path_config, _PROJECT_ROOT

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
pc = load_path_config(config_name)

PATIENT_DATA_DIR = pc.working_dir / "patient_data"
DS_FOLDER = "downsampled_full_fov_128x128x64_crop-17.5"

# Reuse the results_df from velocity_direction_check if available,
# otherwise determine original direction inline
VZ_FLOW_TAG = 5

splits_df = pd.read_csv(_PROJECT_ROOT / "splits" / "splits_01-15-26.csv")
patients_df = splits_df[splits_df["split"].isin(["train", "validation", "test"])].copy()
patients_df = patients_df.sort_values(["split", "patient_id"]).reset_index(drop=True)

print(f"Non-skipped patients: {len(patients_df)}")

Found project root at: /home/ayeluru/vascular-superenhancement-4d-flow
Non-skipped patients: 209


In [2]:
def load_vol(path):
    return nib.load(str(path)).get_fdata(dtype=np.float32)


def get_orig_direction(pid):
    """Determine original slice direction from DICOM catalog."""
    catalog_path = PATIENT_DATA_DIR / pid / f"dicom_catalog_{pid}.csv"
    if not catalog_path.exists():
        return "UNKNOWN"
    catalog = pd.read_csv(catalog_path)
    vz_cat = catalog[catalog["tag_0x0043_0x1030"] == VZ_FLOW_TAG].copy()
    if len(vz_cat) == 0:
        return "UNKNOWN"
    vz_cat["time_index"] = (vz_cat["instancenumber"] - 1) % vz_cat["cardiacnumberofimages"]
    vz_cat["slice_index"] = (vz_cat["instancenumber"] - 1) // vz_cat["cardiacnumberofimages"]
    vz_cat["z"] = vz_cat["imagepositionpatient"].apply(lambda x: np.array(eval(x))[2])
    t0 = vz_cat[vz_cat["time_index"] == 0].sort_values("slice_index")
    z_diff = np.diff(t0["z"].values)
    if np.sum(z_diff > 0) > np.sum(z_diff < 0):
        return "I_to_S"
    elif np.sum(z_diff < 0) > np.sum(z_diff > 0):
        return "S_to_I"
    return "AMBIGUOUS"


rows = []

for i, (_, patient_row) in enumerate(patients_df.iterrows()):
    pid = patient_row["patient_id"]
    split = patient_row["split"]
    ds_root = PATIENT_DATA_DIR / pid / "nifti" / DS_FOLDER

    paths = {
        comp: (ds_root / f"4d_flow_{comp}" / f"4d_flow_{comp}_{pid}_frame_00.nii.gz",
               ds_root / f"4d_flow_{comp}_corr" / f"4d_flow_{comp}_corr_{pid}_frame_00.nii.gz")
        for comp in ["vx", "vy", "vz"]
    }
    mag_path = ds_root / "4d_flow_mag" / f"4d_flow_mag_{pid}_frame_00.nii.gz"

    if not all(p.exists() for pair in paths.values() for p in pair) or not mag_path.exists():
        print(f"SKIP {pid}: missing files")
        continue

    mag = load_vol(mag_path)
    mag_thresh = np.percentile(mag[mag > 0], 5) if np.any(mag > 0) else 0
    mask = mag > mag_thresh

    uncorr = np.stack([load_vol(paths[c][0]) for c in ["vx", "vy", "vz"]], axis=-1)  # (X,Y,Z,3)
    corr = np.stack([load_vol(paths[c][1]) for c in ["vx", "vy", "vz"]], axis=-1)

    # Distance as-is
    d_as_is = np.linalg.norm(uncorr - corr, axis=-1)

    # Distance with corrected vz negated
    corr_neg_vz = corr.copy()
    corr_neg_vz[..., 2] = -corr_neg_vz[..., 2]
    d_neg_vz = np.linalg.norm(uncorr - corr_neg_vz, axis=-1)

    # Stats within mask
    mean_as_is = d_as_is[mask].mean()
    mean_neg_vz = d_neg_vz[mask].mean()

    orig_dir = get_orig_direction(pid)

    rows.append({
        "patient_id": pid,
        "split": split,
        "original_direction": orig_dir,
        "mean_d_as_is": mean_as_is,
        "mean_d_neg_vz": mean_neg_vz,
        "vz_sign_correct": mean_as_is < mean_neg_vz,
        "ratio": mean_as_is / mean_neg_vz if mean_neg_vz > 0 else np.nan,
    })
    print(f"[{i+1}/{len(patients_df)}] {pid}: d_as_is={mean_as_is:.1f}  d_neg_vz={mean_neg_vz:.1f}  "
          f"{'OK' if mean_as_is < mean_neg_vz else 'WRONG VZ SIGN'}  ({orig_dir})")

dist_df = pd.DataFrame(rows)
print("\nDone.")

[1/209] Balboloop: d_as_is=73.0  d_neg_vz=329.2  OK  (I_to_S)
[2/209] Biswifo: d_as_is=85.5  d_neg_vz=202.9  OK  (S_to_I)
[3/209] Bomatog: d_as_is=81.5  d_neg_vz=188.8  OK  (S_to_I)
[4/209] Boochuto: d_as_is=75.0  d_neg_vz=163.8  OK  (S_to_I)
[5/209] Boumorim: d_as_is=75.6  d_neg_vz=141.1  OK  (S_to_I)
[6/209] Bovutou: d_as_is=90.6  d_neg_vz=631.2  OK  (I_to_S)
[7/209] Cadotueg: d_as_is=81.5  d_neg_vz=156.1  OK  (S_to_I)
[8/209] Detodu: d_as_is=80.8  d_neg_vz=153.7  OK  (I_to_S)
[9/209] Diecudey: d_as_is=89.6  d_neg_vz=444.7  OK  (I_to_S)
[10/209] Diepami: d_as_is=73.4  d_neg_vz=236.1  OK  (S_to_I)
[11/209] Diequipi: d_as_is=88.1  d_neg_vz=226.5  OK  (S_to_I)
[12/209] Dithigog: d_as_is=93.7  d_neg_vz=186.7  OK  (S_to_I)
[13/209] Dublafer: d_as_is=54.9  d_neg_vz=214.6  OK  (I_to_S)
[14/209] Dujomal: d_as_is=62.7  d_neg_vz=201.0  OK  (S_to_I)
[15/209] Elagieg: d_as_is=67.4  d_neg_vz=157.0  OK  (I_to_S)
[16/209] Golotag: d_as_is=67.6  d_neg_vz=227.5  OK  (S_to_I)
[17/209] Grequafie: d_as_

In [3]:
print("=" * 80)
print("SUMMARY: vz sign correctness")
print("  vz_sign_correct=True  → as-is distance smaller → current vz sign is right")
print("  vz_sign_correct=False → negated distance smaller → vz has WRONG sign")
print("=" * 80)

print(f"\nTotal OK:    {dist_df['vz_sign_correct'].sum()}/{len(dist_df)}")
print(f"Total WRONG: {(~dist_df['vz_sign_correct']).sum()}/{len(dist_df)}")

print("\n--- Cross-tab: Original direction vs vz_sign_correct ---")
print(pd.crosstab(dist_df["original_direction"], dist_df["vz_sign_correct"],
                  colnames=["vz_sign_correct"], margins=True))

wrong = dist_df[~dist_df["vz_sign_correct"]]
if len(wrong) > 0:
    print(f"\n--- Patients with WRONG vz sign ({len(wrong)}) ---")
    print(wrong[["patient_id", "original_direction", "mean_d_as_is", "mean_d_neg_vz", "ratio"]].to_string(index=False))

SUMMARY: vz sign correctness
  vz_sign_correct=True  → as-is distance smaller → current vz sign is right
  vz_sign_correct=False → negated distance smaller → vz has WRONG sign

Total OK:    208/208
Total WRONG: 0/208

--- Cross-tab: Original direction vs vz_sign_correct ---
vz_sign_correct     True  All
original_direction           
I_to_S               116  116
S_to_I                92   92
All                  208  208


In [4]:
with pd.option_context("display.max_rows", None, "display.float_format", "{:.2f}".format):
    display(dist_df.sort_values(["vz_sign_correct", "original_direction", "patient_id"]))

,patient_id,split,original_direction,mean_d_as_is,mean_d_neg_vz,vz_sign_correct,ratio
29,Aruborn,train,I_to_S,66.58,171.82,True,0.39
30,Asonlig,train,I_to_S,125.55,241.95,True,0.52
31,Badiswu,train,I_to_S,124.67,285.33,True,0.44
0,Balboloop,test,I_to_S,73.01,329.20,True,0.22
32,Bibathot,train,I_to_S,42.67,251.41,True,0.17
34,Boudubat,train,I_to_S,54.51,301.16,True,0.18
5,Bovutou,test,I_to_S,90.57,631.15,True,0.14
36,Butiswu,train,I_to_S,71.15,204.47,True,0.35
37,Cadedag,train,I_to_S,58.64,182.29,True,0.32
38,Cefaru,train,I_to_S,110.02,227.26,True,0.48
